# StabilityArc: sequence to mutation scores
Run this notebook in a Google Colab GPU runtime. It downloads the StabilityArc release from Hugging Face; the first prediction also downloads the separately hosted frozen ESMC-600M encoder. Internet access and several GB of memory are required. The released all-66 checkpoint is for inference on external proteins; it does not reproduce protein-held-out ProteinGym results.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub>=1.0', 'pandas'])
from huggingface_hub import snapshot_download

repo_dir = snapshot_download(
    'aaronfeller/StabilityArc',
    allow_patterns=['stabilityarc/*.py', 'checkpoints/stabilityarc.pt', 'pyproject.toml'],
)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', repo_dir])


## Predict all single substitutions
Enter a wild-type sequence using the 20 standard amino acids. Positions in the output are 1-based.

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from stabilityarc.inference import StabilityArcPredictor

sequence = 'ACDEFGHIKLMNPQRSTVWY'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
predictor = StabilityArcPredictor(Path(repo_dir) / 'checkpoints' / 'stabilityarc.pt', device=device)
predictions = pd.DataFrame(predictor.predict(sequence))
predictions.head(20)

In [ ]:
predictions.to_csv('stabilityarc_predictions.csv', index=False)
print(f'Saved {len(predictions)} substitutions to stabilityarc_predictions.csv')

## Interpreting predictions
Each site has 19 non-wild-type substitutions. Larger `stabilityarc_stability_score` means predicted greater stability. `stabilityarc_predicted_ddG` is its negative, so larger values mean predicted destabilization. These are model scores, not calibrated experimental kcal/mol measurements. This notebook scores single substitutions; multi-mutation residuals require attention-derived contact maps.